# Quinta lista de ejercicios – Selección de variables y generalización

**Introducción al Machine Learning – Universidad El Bosque – 2026–2**

Este cuaderno resuelve los cuatro ejercicios usando `Datos_Hitters.csv`. Ejecute las celdas en orden, desde un kernel reiniciado.

## Antes de comenzar: qué se va a hacer

1. Separar los datos en entrenamiento (50), validación (70) y prueba (143), respetando `random_state=31`.
2. Comparar regresión lineal completa, selección por correlación, selección por pruebas t y selección hacia adelante.
3. Escoger el modelo con menor riesgo de validación.
4. Evaluar los cuatro modelos en prueba, que se mantiene reservada hasta el ejercicio 4.

**Decisión sobre `index`:** el archivo tiene 21 columnas: `index`, 19 predictoras y `Salary`. `index` es un identificador de fila, por lo que no se usa como predictor. La respuesta es `Salary`.

In [ ]:
# Importar las librerías y las funciones del proyecto
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm

# Encontrar la carpeta raíz del proyecto, incluso si el notebook se abre desde notebooks/
cwd = Path.cwd()
if (cwd / "data" / "Datos_Hitters.csv").exists():
    PROJECT_DIR = cwd
elif (cwd.parent / "data" / "Datos_Hitters.csv").exists():
    PROJECT_DIR = cwd.parent
else:
    raise FileNotFoundError("No se encontró data/Datos_Hitters.csv. Ejecute desde la carpeta del proyecto.")

sys.path.insert(0, str(PROJECT_DIR / "src"))
from analisis import riesgo_cuadratico, ajustar_y_evaluar, seleccionar_hacia_adelante, obtener_pvalores

DATA_PATH = PROJECT_DIR / "data" / "Datos_Hitters.csv"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Dimensiones:", df.shape)
display(df.head())

## 1. Preparación y partición de los datos

El enunciado exige dos particiones consecutivas:

- Primera etapa: 120 observaciones para desarrollo y 143 para prueba.
- Segunda etapa: de las 120 de desarrollo, 50 para entrenamiento y 70 para validación.

Se usa `train_size` para indicar explícitamente cuántas observaciones quedan en cada conjunto. En ambas etapas se fija `random_state=31`.

No se utiliza el conjunto de prueba para seleccionar variables ni ajustar decisiones en los ejercicios 1–3.

In [ ]:
# Identificar respuesta y predictores
respuesta = "Salary"
variables = [col for col in df.columns if col not in ["index", respuesta]]

print("Número de predictores:", len(variables))
print("Predictores:", variables)

# Primera etapa: desarrollo = 120; prueba = 143
desarrollo, prueba = train_test_split(
    df, train_size=120, random_state=31
)

# Segunda etapa: entrenamiento = 50; validación = 70
entrenamiento, validacion = train_test_split(
    desarrollo, train_size=50, random_state=31
)

print("\\nTamaños:")
print("Entrenamiento:", len(entrenamiento))
print("Validación:", len(validacion))
print("Desarrollo:", len(desarrollo))
print("Prueba:", len(prueba))

X_train = entrenamiento[variables]
y_train = entrenamiento[respuesta]
X_valid = validacion[variables]
y_valid = validacion[respuesta]
X_test = prueba[variables]
y_test = prueba[respuesta]

assert (len(entrenamiento), len(validacion), len(prueba)) == (50, 70, 143)

## Función de riesgo cuadrático

La guía define el riesgo sobre un conjunto \(S\) como:

\[
R_S(h)=\frac{1}{|S|}\sum_{i\in S}(y_i-h(x_i))^2.
\]

Esto es el error cuadrático medio (MSE). Se implementa una función propia, tal como solicita el enunciado. Un valor menor indica menor error promedio cuadrático en ese conjunto.

In [ ]:
# Comprobación de la función propia
def riesgo(y_real, y_estimado):
    y_real = np.asarray(y_real)
    y_estimado = np.asarray(y_estimado)
    return np.mean((y_real - y_estimado) ** 2)

print("La función de riesgo está lista.")

## Ejercicio 1. Modelo completo

Se ajusta una regresión lineal con intercepto utilizando las 19 variables predictoras.

- El riesgo de entrenamiento se calcula sobre las 50 observaciones de entrenamiento.
- El riesgo de validación se calcula sobre las 70 observaciones de validación.

In [ ]:
modelo_completo, riesgo_train_completo, riesgo_valid_completo = ajustar_y_evaluar(
    X_train, y_train, X_valid, y_valid, variables
)

tabla_ejercicio_1 = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Validación"],
    "Riesgo cuadrático": [riesgo_train_completo, riesgo_valid_completo]
})
display(tabla_ejercicio_1.round(2))
tabla_ejercicio_1.to_csv(RESULTS_DIR / "ejercicio_1_modelo_completo.csv", index=False)

**Interpretación del ejercicio 1.** El modelo completo obtiene un riesgo de entrenamiento mucho menor que el de validación. Esta diferencia es una señal de que el modelo puede estar ajustándose demasiado a las 50 observaciones usadas para entrenarlo. Por eso no basta con mirar el riesgo de entrenamiento: también se necesita validación para estimar cómo se comporta con datos no usados en el ajuste.

## Ejercicio 2. Selección estadística de variables

### 2(a). Correlación de Pearson

Se calcula la correlación de Pearson entre cada predictor y `Salary`, usando solamente entrenamiento. Después se ordena por el valor absoluto de la correlación, de mayor a menor. El valor absoluto determina la fuerza de la asociación lineal, sin importar si es positiva o negativa.

In [ ]:
correlaciones = X_train.corrwith(y_train).sort_values(
    key=lambda valores: valores.abs(), ascending=False
)
tabla_correlaciones = pd.DataFrame({
    "Correlación con Salary": correlaciones,
    "Valor absoluto": correlaciones.abs()
})
display(tabla_correlaciones.round(4))
tabla_correlaciones.to_csv(RESULTS_DIR / "ejercicio_2a_correlaciones.csv")

### 2(b). Regresión con las siete variables de mayor correlación

Se seleccionan las primeras siete variables del orden anterior. Se ajusta una regresión lineal con intercepto y se calculan los dos riesgos.

In [ ]:
variables_top7 = list(correlaciones.head(7).index)
print("Siete variables seleccionadas:", variables_top7)

modelo_top7, riesgo_train_top7, riesgo_valid_top7 = ajustar_y_evaluar(
    X_train, y_train, X_valid, y_valid, variables_top7
)

print(f"Riesgo entrenamiento: {riesgo_train_top7:.2f}")
print(f"Riesgo validación:   {riesgo_valid_top7:.2f}")

### 2(c). Selección mediante pruebas t

Primero se ajusta el modelo lineal completo mediante `statsmodels`, para obtener los p-valores de las pruebas t individuales de los coeficientes. Se excluye el intercepto de la selección y se conservan los predictores cuyo p-valor sea estrictamente menor que 0,05.

Luego se ajusta una nueva regresión lineal con las variables seleccionadas y se calculan los riesgos de entrenamiento y validación.

In [ ]:
ajuste_ols, p_valores = obtener_pvalores(X_train, y_train)
tabla_pvalores = pd.DataFrame({
    "Valor p": p_valores
}).sort_values("Valor p")

display(tabla_pvalores.round(6))
tabla_pvalores.to_csv(RESULTS_DIR / "ejercicio_2c_pvalores.csv")

variables_significativas = list(p_valores[p_valores < 0.05].index)
print("Variables con p < 0.05:", variables_significativas)

modelo_t, riesgo_train_t, riesgo_valid_t = ajustar_y_evaluar(
    X_train, y_train, X_valid, y_valid, variables_significativas
)

print(f"Riesgo entrenamiento: {riesgo_train_t:.2f}")
print(f"Riesgo validación:   {riesgo_valid_t:.2f}")

### 2(d). Comparación de los criterios

La correlación selecciona las variables que individualmente tienen mayor asociación lineal con `Salary`. Las pruebas t, en cambio, evalúan cada coeficiente dentro del modelo completo, controlando por las demás variables incluidas. Por eso los dos procedimientos no tienen por qué seleccionar el mismo conjunto.

Un p-valor pequeño no mide el tamaño del efecto ni garantiza por sí solo mejor predicción. La comparación predictiva se hace con los riesgos de validación.

In [ ]:
tabla_ejercicio_2 = pd.DataFrame([
    ["Mayor correlación", len(variables_top7), ", ".join(variables_top7), riesgo_train_top7, riesgo_valid_top7],
    ["Pruebas t (p < 0.05)", len(variables_significativas), ", ".join(variables_significativas), riesgo_train_t, riesgo_valid_t],
], columns=["Criterio", "Número de variables", "Variables", "Riesgo entrenamiento", "Riesgo validación"])

display(tabla_ejercicio_2.round(2))
tabla_ejercicio_2.to_csv(RESULTS_DIR / "ejercicio_2d_comparacion.csv", index=False)

## Ejercicio 3. Selección hacia adelante

El procedimiento comienza probando cada predictor por separado y elige el que produzca el menor riesgo de entrenamiento. Después, en cada paso, prueba agregar cada variable que todavía no esté seleccionada y agrega aquella que más reduzca el riesgo de entrenamiento. Las variables elegidas nunca se eliminan.

Se continúa hasta incorporar las 19 variables. En cada paso también se calcula el riesgo de validación, pero este no se usa para decidir qué variable agregar; se utiliza al final para elegir el tamaño del modelo.

In [ ]:
tabla_forward, mejor_numero_forward, modelo_forward, variables_forward = seleccionar_hacia_adelante(
    X_train, y_train, X_valid, y_valid, variables
)

display(tabla_forward.drop(columns=["Variables"]).round(2))
tabla_forward.to_csv(RESULTS_DIR / "ejercicio_3_seleccion_hacia_adelante.csv", index=False)

print("Número de variables con menor riesgo de validación:", mejor_numero_forward)
print("Variables del modelo elegido:", variables_forward)
mejor_fila = tabla_forward.iloc[mejor_numero_forward - 1]
print(f"Riesgo entrenamiento: {mejor_fila['Riesgo entrenamiento']:.2f}")
print(f"Riesgo validación:   {mejor_fila['Riesgo validación']:.2f}")

### 3(b). Gráfica de riesgos

La gráfica permite observar cómo cambian los riesgos al aumentar el número de variables. El mínimo de la curva de validación determina el tamaño seleccionado.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(tabla_forward["Número de variables"], tabla_forward["Riesgo entrenamiento"],
         marker="o", label="Entrenamiento")
plt.plot(tabla_forward["Número de variables"], tabla_forward["Riesgo validación"],
         marker="o", label="Validación")
plt.axvline(mejor_numero_forward, linestyle="--",
            label=f"Menor validación: {mejor_numero_forward} variables")
plt.xlabel("Número de variables")
plt.ylabel("Riesgo cuadrático (MSE)")
plt.title("Selección hacia adelante: riesgo vs. número de variables")
plt.xticks(range(1, 20))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ejercicio_3_riesgos_forward.png", dpi=160)
plt.show()

### 3(c). Modelo elegido

Se elige el modelo del paso cuyo riesgo de validación sea el menor entre los 19 modelos construidos. Esta regla utiliza validación para escoger el número de variables, no el conjunto de prueba.

In [ ]:
print("Modelo elegido por selección hacia adelante:")
for variable in variables_forward:
    print("-", variable)

## Ejercicio 4. Comparación final

Primero se elige uno de los cuatro procedimientos según su riesgo de validación. Después, y solamente en este ejercicio, se evalúan los cuatro modelos en las 143 observaciones de prueba.

El modelo seleccionado por validación se mantiene fijo; no se vuelve a seleccionar usando los resultados de prueba.

In [ ]:
# Riesgos en prueba: evaluación final, sin reentrenar los modelos
modelos_finales = {
    "Todas las variables": (modelo_completo, variables),
    "Mayor correlación": (modelo_top7, variables_top7),
    "Pruebas t": (modelo_t, variables_significativas),
    "Selección hacia adelante": (modelo_forward, variables_forward),
}

filas_finales = []
for nombre, (modelo, columnas) in modelos_finales.items():
    pred_train = modelo.predict(X_train[columnas])
    pred_valid = modelo.predict(X_valid[columnas])
    pred_test = modelo.predict(X_test[columnas])

    filas_finales.append({
        "Modelo": nombre,
        "Variables": len(columnas),
        "Riesgo train": riesgo(y_train, pred_train),
        "Riesgo validación": riesgo(y_valid, pred_valid),
        "Riesgo prueba": riesgo(y_test, pred_test),
        "Lista de variables": ", ".join(columnas)
    })

tabla_final = pd.DataFrame(filas_finales)
display(tabla_final.drop(columns=["Lista de variables"]).round(2))
tabla_final.to_csv(RESULTS_DIR / "ejercicio_4_comparacion_final.csv", index=False)

modelo_elegido = tabla_final.loc[tabla_final["Riesgo validación"].idxmin()]
print("\\nModelo elegido por validación:", modelo_elegido["Modelo"])
print("Riesgo de validación:", round(modelo_elegido["Riesgo validación"], 2))
print("Variables:", modelo_elegido["Lista de variables"])

### 4(c) y 4(d). Respuestas e interpretación

- **Menor riesgo de entrenamiento:** se identifica buscando el mínimo de la columna `Riesgo train`.
- **Menor riesgo de prueba:** se identifica buscando el mínimo de la columna `Riesgo prueba`.
- **¿Coinciden?** Se compara si esos mínimos corresponden al mismo modelo.
- **¿El modelo escogido con validación fue el de menor riesgo de prueba?** Se compara el modelo con menor validación con el modelo que tiene menor prueba.

El riesgo de prueba es una evaluación final de generalización para esta partición. No se debe usar para cambiar el modelo elegido, porque dejaría de ser un conjunto reservado.

In [ ]:
menor_train = tabla_final.loc[tabla_final["Riesgo train"].idxmin()]
menor_test = tabla_final.loc[tabla_final["Riesgo prueba"].idxmin()]
menor_valid = tabla_final.loc[tabla_final["Riesgo validación"].idxmin()]

print("Menor riesgo de entrenamiento:", menor_train["Modelo"],
      f"({menor_train['Riesgo train']:.2f})")
print("Menor riesgo de prueba:", menor_test["Modelo"],
      f"({menor_test['Riesgo prueba']:.2f})")
print("Menor riesgo de validación:", menor_valid["Modelo"],
      f"({menor_valid['Riesgo validación']:.2f})")
print("¿Validación eligió el menor riesgo de prueba?:",
      menor_valid["Modelo"] == menor_test["Modelo"])

## Conclusiones generales

1. El modelo completo presenta un riesgo de entrenamiento bajo, pero un riesgo de validación muy alto. Esto muestra por qué el desempeño en entrenamiento no es suficiente para elegir un modelo.
2. La selección por correlación y la selección por pruebas t producen subconjuntos diferentes, porque responden a criterios estadísticos distintos.
3. La selección hacia adelante permite comparar el riesgo de validación para distintos tamaños de subconjunto y elegir el tamaño que mejor generaliza dentro del conjunto de validación.
4. La comparación final en prueba permite revisar cómo se comportaron los cuatro procedimientos en datos no usados durante el desarrollo. La prueba no debe convertirse en otro criterio de ajuste.
5. Todos los riesgos reportados son MSE y están expresados en las unidades de `Salary` al cuadrado.

## Archivos generados

Al ejecutar todo el cuaderno se guardan en `results/` las tablas CSV y la gráfica solicitada.